# 05 — PyTorch on CPU, then PyTorch/XLA TPU-ready

**Learning goal.** Run a tiny MLP forward pass in PyTorch on CPU, then see the minimal change you'd make to target a Cloud TPU with **PyTorch/XLA**.

**What you'll observe.**
- A tiny MLP defined with `torch.nn.Linear`.
- A CPU forward pass timed for a steady-state estimate.
- A guarded "TPU-ready" cell using `torch_xla.core.xla_model` (`xm.xla_device()`, `xm.mark_step()`).

**Knobs to tune.**
- `BATCH`, `HIDDEN`, `N_LAYERS`.
- On a TPU VM, the `PJRT_DEVICE=TPU` env var selects the PJRT backend.

**Failure modes to watch.**
- `torch_xla` is typically not present on a vanilla local Python install — the guarded cell prints a notice and skips.
- The first call after `xm.xla_device()` traces & compiles. Warm up before timing.
- Forgetting `xm.mark_step()` queues operations indefinitely without execution.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## CPU PyTorch MLP

Standard `torch.nn.Module`. Falls back gracefully if torch is missing.

In [ ]:
try:
    import torch
    from torch import nn
    HAVE_TORCH = True
except Exception as e:
    HAVE_TORCH = False
    print("PyTorch is not installed in this kernel:", e)
    print("Skipping torch cells.")

In [ ]:
if HAVE_TORCH:
    print("torch version:", torch.__version__)
    BATCH, IN_DIM, HIDDEN, OUT, N_LAYERS = 8, 64, 128, 10, 3

    class TinyMLP(nn.Module):
        def __init__(self, in_dim, hidden, out, n_layers):
            super().__init__()
            layers = []
            d = in_dim
            for i in range(n_layers - 1):
                layers += [nn.Linear(d, hidden), nn.ReLU()]
                d = hidden
            layers.append(nn.Linear(d, out))
            self.net = nn.Sequential(*layers)
        def forward(self, x):
            return self.net(x)

    model = TinyMLP(IN_DIM, HIDDEN, OUT, N_LAYERS)
    x = torch.randn(BATCH, IN_DIM)
    print("model output shape:", model(x).shape)
else:
    print("(skipped — no PyTorch)")

In [ ]:
if HAVE_TORCH:
    import time
    model.eval()
    with torch.no_grad():
        # Warm-up
        for _ in range(3):
            _ = model(x)
        t0 = time.perf_counter()
        for _ in range(50):
            _ = model(x)
        elapsed = (time.perf_counter() - t0) / 50
    print(f"CPU forward avg {elapsed*1000:.3f} ms/call")
else:
    print("(skipped — no PyTorch)")

## TPU-ready (only runs on Cloud TPU)

PyTorch/XLA is a thin layer that traces PyTorch ops into XLA. Two idioms matter:

- `xm.xla_device()` returns the current XLA device (a TPU core, or CPU if `PJRT_DEVICE=CPU`).
- `xm.mark_step()` flushes the queued lazy graph and triggers compile + execute. Without it, ops accumulate forever.

The cell is guarded so it's safe to run on a machine without `torch_xla`.

In [ ]:
# TPU-ready (only runs on Cloud TPU; safely skips if torch_xla is missing)
HAVE_XLA = False
if HAVE_TORCH:
    try:
        import torch_xla.core.xla_model as xm
        HAVE_XLA = True
    except Exception as e:
        print("torch_xla is not installed:", e)
        print("On a Cloud TPU VM, install with: pip install torch_xla[tpu]")

if HAVE_TORCH and HAVE_XLA:
    device = xm.xla_device()
    print("xla device:", device)
    model_xla = TinyMLP(IN_DIM, HIDDEN, OUT, N_LAYERS).to(device)
    x_xla = x.to(device)

    # First call: traces and compiles. The mark_step is what executes the queued graph.
    out = model_xla(x_xla)
    xm.mark_step()
    print("output device:", out.device, "shape:", out.shape)

    # Steady-state
    import time
    for _ in range(3):
        _ = model_xla(x_xla); xm.mark_step()
    t0 = time.perf_counter()
    for _ in range(20):
        _ = model_xla(x_xla); xm.mark_step()
    elapsed = (time.perf_counter() - t0) / 20
    print(f"XLA forward avg {elapsed*1000:.3f} ms/call")
else:
    print("(skipped — no torch_xla in this kernel)")

## What changes on a real Cloud TPU VM?

- `xm.xla_device()` returns a `xla:0` device backed by a real TPU core.
- The first forward pass for each unique input shape pays an XLA compile cost.
- Multi-device parallel training uses `torch_xla.distributed.xla_multiprocessing.spawn` to launch one process per TPU core, plus `xm.optimizer_step(optimizer)` which inserts an all-reduce.
- Always finish a training epoch with `xm.mark_step()` so the last batch executes.

## Exercises

1. On CPU, time the MLP at `HIDDEN=128, 512, 2048`. Plot ms/call vs hidden size. Where does it stop scaling linearly?
2. Add `optimizer = torch.optim.Adam(model.parameters())` and a training loop. Time one CPU step.
3. On a TPU VM, run the XLA cell and compare to CPU. What fraction of the first call is compile time?
4. What happens if you remove `xm.mark_step()`? Use `xm.get_xla_compute_runtime_object()` or simply `nvidia-smi`-style monitoring to confirm nothing executed yet.
5. Wrap the forward pass in `torch.compile(model)` (PyTorch 2.x) and compare CPU times. Note that `torch.compile` is **not** torch_xla; they share goals but different stacks.